<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_07_protein_3d_conformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Folding the Molecular Machine (Protein 3D Conformation)**

In this lecture, we move from sequences to **3D structures**. Proteins are molecular machines whose function is completely determined by their folded conformation.

- **Biological Mechanism:** Ribosomal translation produces a linear polypeptide. However, peptide bond planar resonance and steric hindrance severely constrain the backbone flexibility (quantified by Ramachandran $\phi$ and $\psi$ angles). Driven by hydrophobic collapse and secondary structure nucleation ($\alpha$-helices, $\beta$-sheets), the chain folds into a precise 3D shape.
- **Instrumental Method:** We determine these atomic coordinates experimentally using X-ray crystallography (solving electron density phases) and, increasingly, Cryogenic Electron Microscopy (Cryo-EM).
- **Underlying Algorithm:** To compare structures, we use Spatial Coordinate Transformations & the **Kabsch Algorithm**. By finding the optimal rotation and translation via Singular Value Decomposition (SVD), we minimize the Root-Mean-Square Deviation (RMSD) between two aligned structures:
  $$\text{RMSD} = \sqrt{\frac{1}{N} \sum_{i=1}^N \Vert{}\mathbf{x}_i - \mathbf{y}_i\Vert{}^2}$$
- **Industrial Tool:** `Bio.PDB` for parsing structural data and `py3Dmol` for WebGL in-notebook rendering.
- **Dataset:** SARS-CoV-2 Main Protease (Mpro) wild-type (6LU7) and mutant structures.


### **Part 1: Environment Setup**

We install Biopython to parse `.pdb` files and `py3Dmol` to visualize the 3D structures directly in the notebook.


In [ ]:

import os
from IPython.display import clear_output

# Install Biopython, py3Dmol, and standard libraries
!pip install -q biopython py3Dmol numpy pandas matplotlib scipy

clear_output()
print("Installations successful.")

### **Part 2: Dataset Download**

We will download the SARS-CoV-2 Main Protease structure (PDB ID: 6LU7) directly from the RCSB Protein Data Bank.


In [ ]:
import urllib.request

pdb_id = "6LU7"
pdb_filename = f"{pdb_id}.pdb"

if not os.path.exists(pdb_filename):
  print(f"Downloading {pdb_id} from RCSB PDB...")
  url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
  # Using Request with User-Agent to prevent HTTP 403 blocks
  request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
  with urllib.request.urlopen(request) as response, open(pdb_filename, "wb") as out_file:
    out_file.write(response.read())

clear_output()
print(f"Dataset {pdb_filename} available.")


### **Part 3: Parsing the PDB File**

The `.pdb` format describes the 3D coordinates ($x, y, z$) of every atom in the molecule. We use `Bio.PDB.PDBParser` to read it into a hierarchical data structure:
**Structure $\rightarrow$ Model $\rightarrow$ Chain $\rightarrow$ Residue $\rightarrow$ Atom**.


In [ ]:
from Bio.PDB import PDBParser
import warnings

# Suppress PDBConstructionWarnings for non-standard residues in this PDB
from Bio.PDB.PDBExceptions import PDBConstructionWarning
warnings.simplefilter('ignore', PDBConstructionWarning)

parser = PDBParser()
structure = parser.get_structure(pdb_id, pdb_filename)

# A PDB file can have multiple models (e.g. from NMR), but X-ray structures usually have one (Model 0).
model = structure[0]

# Chain A is typically the main protein polymer
chain_A = model['A']

# Let's count how many amino acid residues are in Chain A
residues = [res for res in chain_A if res.id[0] == ' '] # Filter out heteroatoms (like water, ligands)
print(f"Structure {pdb_id} parsed.")
print(f"Chain A contains {len(residues)} standard amino acid residues.")

# Look at the coordinates of the Alpha Carbon (CA) of the first residue
first_res = residues[0]
print(f"First residue: {first_res.resname} {first_res.id[1]}")
print(f"CA coordinates: {first_res['CA'].get_coord()}")


### **Part 4: The Ramachandran Plot**

The peptide bond is planar, meaning the backbone only has two degrees of freedom per residue:
- $\phi$ (phi): angle of rotation about the N-CA bond.
- $\psi$ (psi): angle of rotation about the CA-C bond.

Steric hindrance (atoms crashing into each other) prevents most ($\phi, \psi$) combinations. The allowed regions correspond to secondary structures like $\alpha$-helices and $\beta$-sheets. Let's calculate and plot them!


In [ ]:
from Bio.PDB.Polypeptide import PPBuilder
import math
import matplotlib.pyplot as plt

builder = PPBuilder()
polypeptides = builder.build_peptides(chain_A)

phi_angles = []
psi_angles = []

for poly in polypeptides:
  # get_phi_psi_list returns tuples of (phi, psi) in radians
  angles = poly.get_phi_psi_list()
  for phi, psi in angles:
    if phi is not None and psi is not None:
      phi_angles.append(math.degrees(phi))
      psi_angles.append(math.degrees(psi))

plt.figure(figsize=(6, 6))
plt.scatter(phi_angles, psi_angles, s=10, alpha=0.5, color='tab:blue')
plt.title(f"Ramachandran Plot of {pdb_id}")
plt.xlabel(r"$\phi$ (degrees)")
plt.ylabel(r"$\psi$ (degrees)")
plt.xlim(-180, 180)
plt.ylim(-180, 180)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)

# Highlight typical alpha-helix and beta-sheet regions loosely
plt.axhspan(-60, -40, xmin=0.2, xmax=0.4, color='red', alpha=0.1, label='Right-handed α-helix')
plt.axhspan(100, 160, xmin=0.1, xmax=0.4, color='green', alpha=0.1, label='β-sheet')

plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()


### **Part 5: Kabsch Algorithm & RMSD Superposition**

How do we quantitatively compare two protein structures? We align them in 3D space.
The **Kabsch algorithm** finds the optimal rotation matrix to minimize the Root-Mean-Square Deviation (RMSD) between two sets of paired points.

Steps:
1. Translate both sets of coordinates so their centroids are at the origin.
2. Compute the cross-covariance matrix $C = X^T Y$.
3. Compute the SVD of $C$: $C = U S V^T$.
4. The optimal rotation matrix is $R = V U^T$. (With a check to ensure $\det(R) = 1$, preventing reflections).

Biopython's `Superimposer` implements this internally. Let's compare 6LU7 (SARS-CoV-2 Mpro) with an older SARS-CoV-1 Mpro structure (1UK4).


In [ ]:
from Bio.PDB import Superimposer

pdb_id_2 = "1UK4"
pdb_filename_2 = f"{pdb_id_2}.pdb"

if not os.path.exists(pdb_filename_2):
  print(f"Downloading {pdb_id_2} from RCSB PDB...")
  url = f"https://files.rcsb.org/download/{pdb_id_2}.pdb"
  request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
  with urllib.request.urlopen(request) as response, open(pdb_filename_2, "wb") as out_file:
    out_file.write(response.read())
  clear_output()

structure_2 = parser.get_structure(pdb_id_2, pdb_filename_2)
chain_A_2 = structure_2[0]['A']

# Extract Alpha Carbon (CA) atoms for alignment
atoms_1 = [res['CA'] for res in chain_A if res.id[0] == ' ' and 'CA' in res]
atoms_2 = [res['CA'] for res in chain_A_2 if res.id[0] == ' ' and 'CA' in res]

# We must align equal numbers of atoms. Here we just take the first N matching atoms for simplicity.
# In a real scenario, you would sequence-align them first.
min_len = min(len(atoms_1), len(atoms_2))
atoms_1_aligned = atoms_1[:min_len]
atoms_2_aligned = atoms_2[:min_len]

super_imposer = Superimposer()
super_imposer.set_atoms(atoms_1_aligned, atoms_2_aligned)
super_imposer.apply(structure_2.get_atoms()) # Apply transformation to the entire second structure

print(f"Superimposing {pdb_id} and {pdb_id_2}")
print(f"Number of atoms aligned: {min_len}")
print(f"RMSD: {super_imposer.rms:.3f} Å")


### **Part 6: Visualizing the 3D Structure**

We can use WebGL via `py3Dmol` to visualize the protein in 3D right inside the notebook.
Let's view our SARS-CoV-2 Mpro (6LU7).


In [ ]:
import py3Dmol

with open(pdb_filename, "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")

# Style the protein backbone as a cartoon, colored by secondary structure
view.setStyle({'cartoon': {'color': 'spectrum'}})

# The 6LU7 structure contains an inhibitor (ligand). Let's show it as sticks.
view.setStyle({'hetflag': True}, {'stick': {}})

view.zoomTo()
view.show()


### **Homework: Protein 3D Conformation**

Use the concepts and tools demonstrated above to solve the following tasks.


#### **Task 1: Kabsch Algorithm from Scratch**

Write a pure NumPy function `kabsch_rmsd(P, Q)` that computes the RMSD between two sets of coordinates after optimal rotation and translation.

$P$ and $Q$ are $N \times 3$ matrices.
1. Compute the centroids of $P$ and $Q$ and center both matrices.
2. Compute the covariance matrix $C = P_{centered}^T Q_{centered}$.
3. Compute the SVD of $C$ using `np.linalg.svd`.
4. Calculate the rotation matrix $R = V U^T$. (Ensure $\det(R) > 0$. If $\det(R) < 0$, flip the sign of the last column of $V$).
5. Rotate $P_{centered}$ and compute the RMSD with $Q_{centered}$.

Test it using the CA coordinates extracted in Part 5. Your RMSD should exactly match `super_imposer.rms`.


In [ ]:
import numpy as np

def kabsch_rmsd(P, Q):
  """
  Returns the RMSD between P and Q after optimal superposition.
  """
  # ---- YOUR CODE HERE ----


# Extract raw coordinates as numpy arrays
P_coords = np.array([atom.get_coord() for atom in atoms_1_aligned])
# We must use the original un-rotated coordinates of structure 2 for this test
raw_structure_2 = parser.get_structure('1UK4_raw', pdb_filename_2)
raw_atoms_2 = [res['CA'] for res in raw_structure_2[0]['A'] if res.id[0] == ' ' and 'CA' in res]
Q_coords = np.array([atom.get_coord() for atom in raw_atoms_2[:min_len]])

custom_rmsd = kabsch_rmsd(P_coords, Q_coords)
print(f"Custom Kabsch RMSD:  {custom_rmsd:.3f} Å")
print(f"Biopython RMSD:      {super_imposer.rms:.3f} Å")

#### **Task 2: Distance Matrix Calculation**

A 2D distance matrix maps the Euclidean distance between every pair of Alpha Carbons in the protein.
Compute the pairwise distance matrix for `chain_A` of 6LU7, and plot it as a heatmap using `matplotlib.pyplot.imshow`.

What visual patterns correspond to $\alpha$-helices? What about parallel vs anti-parallel $\beta$-sheets?


In [ ]:
from scipy.spatial.distance import cdist

# ---- YOUR CODE HERE ----



#### **Task 3: B-factor Analysis**

The B-factor (temperature factor) of an atom indicates its mobility or uncertainty in the crystal lattice. High B-factors usually mean flexible loops, while low B-factors mean rigid core structures.

1. Extract the B-factors for all CA atoms in `chain_A` of 6LU7.
2. Plot them as a line graph (Residue index vs B-factor).
3. Identify the index of the most flexible residue in the chain.

In [ ]:
# ---- YOUR CODE HERE ----


#### **Task 4: Visualizing the Bound Ligand**

The 6LU7 structure contains a peptide-like inhibitor N3. In this specific PDB file, the inhibitor is assigned its own chain (chain C) because it consists of multiple modified amino acid residues, rather than being a single simple heteroatom.
Extract chain C and output how many atoms it has. Then, write a short Py3Dmol script to zoom in *only* on the ligand (chain C).


In [ ]:
# ---- YOUR CODE HERE ----


#### **Task 5: Interpretation Questions**

Answer the following in a text cell:

1. Why do we compute RMSD using the Kabsch algorithm (SVD) rather than simply subtracting coordinate matrices?
2. Looking at the Ramachandran plot, why is the top right quadrant ($\phi > 0, \psi > 0$) mostly empty? What biological exception can reside there?
3. On the distance matrix heatmap, what does a thick band along the main diagonal signify? What would an anti-parallel $\beta$-sheet contact look like?
4. X-ray crystallography requires proteins to form ordered crystals, while Cryo-EM freezes them in a thin layer of vitreous ice. Why might a highly flexible loop (high B-factor) be "invisible" in an X-ray structure?
5. The SVD step in Kabsch is identical to PCA mathematically. However, the geometric meaning is different. In PCA we find axes of maximal variance for *one* dataset; in Kabsch, what exactly is SVD doing with the *cross-covariance* matrix of *two* datasets?
